# Environmental Adaptation Modeling: Conditional Protein Generation

Full-scale GPU-ready training pipeline notebook.

In [ ]:
# !pip install -U pandas numpy pyarrow requests tqdm scikit-learn datasets biopython
# !pip install -U torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
# !pip install -U transformers accelerate peft sentencepiece fair-esm

In [ ]:
import os, json, random, requests
import numpy as np, pandas as pd
from pathlib import Path
from tqdm.auto import tqdm
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
print('DEVICE:', DEVICE)

## 1) Configuration

In [ ]:
CFG={'data_dir':'data','processed_dir':'processed','checkpoints_dir':'checkpoints','min_len':80,'max_len':512,'esm_model_name':'facebook/esm2_t33_650M_UR50D','batch_size':4,'num_workers':4,'lr':2e-4,'weight_decay':1e-2,'epochs':10,'grad_accum':8,'max_train_samples':None,'top_p':0.95,'temperature':0.9}
for k in ['data_dir','processed_dir','checkpoints_dir']: Path(CFG[k]).mkdir(parents=True, exist_ok=True)

## 2) Data sources and accessibility checks

In [ ]:
DATA_SOURCES={'uniprot_rest_example':'https://rest.uniprot.org/uniprotkb/search?query=reviewed:true&format=json&size=1','ncbi_datasets_api_docs':'https://www.ncbi.nlm.nih.gov/datasets/docs/v2/','gold_home':'https://gold.jgi.doe.gov/','nasa_exoplanet_archive':'https://exoplanetarchive.ipac.caltech.edu/','meteoritical_bulletin':'https://www.lpi.usra.edu/meteor/','bacdive':'https://bacdive.dsmz.de/','extremophile_data_paper':'https://www.nature.com/articles/s41597-023-02064-4'}
def check_url(url, timeout=20):
    try:
        r=requests.get(url, timeout=timeout); return r.status_code, len(r.text)
    except Exception as e:
        return None, str(e)
rows=[]
for n,u in DATA_SOURCES.items():
    s,i=check_url(u); rows.append({'source':n,'url':u,'status':s,'info':i})
pd.DataFrame(rows)

## 3) Download connectors (extensible)

In [ ]:
def save_bytes(url,out_path,chunk=1<<20):
    out_path=Path(out_path); out_path.parent.mkdir(parents=True, exist_ok=True)
    with requests.get(url, stream=True, timeout=60) as r:
        r.raise_for_status()
        with open(out_path,'wb') as f:
            for b in r.iter_content(chunk_size=chunk):
                if b: f.write(b)
    return str(out_path)
UNIPROT_SAMPLE_URL=('https://rest.uniprot.org/uniprotkb/search?query=reviewed:true+AND+fragment:false&fields=accession,sequence,length,protein_name,organism_name,cc_function&format=json&size=500')
NASA_EXO_CSV=('https://exoplanetarchive.ipac.caltech.edu/TAP/sync?query=select+top+5000+pl_name,hostname,sy_snum,pl_orbper,pl_rade,pl_bmasse,pl_eqt,st_teff,st_rad,st_mass+from+pscomppars&format=csv')
uniprot_sample_path=Path(CFG['data_dir'])/'uniprot_sample.json'; exo_path=Path(CFG['data_dir'])/'nasa_exoplanets.csv'
if not uniprot_sample_path.exists(): save_bytes(UNIPROT_SAMPLE_URL, uniprot_sample_path)
if not exo_path.exists(): save_bytes(NASA_EXO_CSV, exo_path)
print(uniprot_sample_path, exo_path)

## 4) Environmental features

In [ ]:
ENV_FEATURES=['carbon','nitrogen','oxygen','sulfur','phosphorus','silicon','iron','magnesium','sodium','potassium','chlorine','temperature_c','ph','salinity_psu','pressure_bar','radiation_flux','methane','ammonia','co2']
print('Num condition dims =', len(ENV_FEATURES))

## 5) Build training dataframe

In [ ]:
# Build training dataframe using location-linked environmental metadata (no random condition synthesis)

with open(uniprot_sample_path,'r') as f:
    uni=json.load(f)

records=[]
for r in uni.get('results',[]):
    seq=r.get('sequence',{}).get('value','')
    if not seq or len(seq)<CFG['min_len'] or len(seq)>CFG['max_len']:
        continue
    org=r.get('organism',{})
    records.append({
        'accession': r.get('primaryAccession'),
        'sequence': seq,
        'length': len(seq),
        'organism': org.get('scientificName',''),
        'taxon_id': (org.get('taxonId') if isinstance(org, dict) else None)
    })

df=pd.DataFrame(records)
print('Raw sequence rows:', len(df))

# --- Location-aware metadata join strategy ---
# 1) Query BacDive for taxa/strain-level environment metadata (requires credentials).
# 2) Parse location fields -> lat/lon (+ optional depth/elevation).
# 3) Enrich with Open-Meteo archive climatology by coordinates to derive temperature proxies.
# 4) Map measured fields (pH/salinity/pressure/chemistry) directly when present.

def parse_lat_lon(v):
    if v is None: return None, None
    if isinstance(v, (list, tuple)) and len(v)>=2: return float(v[0]), float(v[1])
    s=str(v).strip()
    # supports forms like '37.1, -122.3'
    parts=[x.strip() for x in s.split(',')]
    if len(parts)>=2:
        try:
            return float(parts[0]), float(parts[1])
        except Exception:
            pass
    return None, None

def fetch_open_meteo_monthly(lat, lon):
    # Annualized temperature proxy for location conditioning
    url=(
        'https://archive-api.open-meteo.com/v1/archive?'
        f'latitude={lat}&longitude={lon}&start_date=2020-01-01&end_date=2020-12-31'
        '&daily=temperature_2m_mean,precipitation_sum'
        '&timezone=UTC'
    )
    try:
        r=requests.get(url, timeout=30)
        r.raise_for_status()
        j=r.json()
        d=j.get('daily', {})
        t=d.get('temperature_2m_mean', []) or []
        p=d.get('precipitation_sum', []) or []
        return {
            'temperature_c': float(np.nanmean(t)) if len(t) else np.nan,
            'precipitation_proxy': float(np.nanmean(p)) if len(p) else np.nan,
        }
    except Exception:
        return {'temperature_c': np.nan, 'precipitation_proxy': np.nan}

# OPTIONAL: set BacDive credentials as env vars before running full pipeline
# export BACDIVE_EMAIL=...
# export BACDIVE_PASSWORD=...

def fetch_bacdive_env_for_taxon(taxon_id):
    # Placeholder connector: implement with BacDive client in authenticated environments.
    # Expected keys returned from BacDive parsed records:
    # lat, lon, ph, salinity_psu, pressure_bar, oxygen, carbon, nitrogen, sulfur, phosphorus, silicon, iron, magnesium, sodium, potassium, chlorine, methane, ammonia, co2
    return {}

rows=[]
for _,r in tqdm(df.iterrows(), total=len(df)):
    env = fetch_bacdive_env_for_taxon(r.get('taxon_id')) if pd.notna(r.get('taxon_id')) else {}

    lat, lon = parse_lat_lon(env.get('lat_lon')) if 'lat_lon' in env else (env.get('lat'), env.get('lon'))
    climate = fetch_open_meteo_monthly(lat, lon) if (lat is not None and lon is not None) else {'temperature_c': np.nan, 'precipitation_proxy': np.nan}

    out = r.to_dict()
    out['lat'] = lat
    out['lon'] = lon

    # measured or derived values only; no random generation
    out['temperature_c'] = env.get('temperature_c', climate['temperature_c'])
    out['ph'] = env.get('ph', np.nan)
    out['salinity_psu'] = env.get('salinity_psu', np.nan)
    out['pressure_bar'] = env.get('pressure_bar', np.nan)
    out['radiation_flux'] = env.get('radiation_flux', np.nan)

    out['carbon'] = env.get('carbon', np.nan)
    out['nitrogen'] = env.get('nitrogen', np.nan)
    out['oxygen'] = env.get('oxygen', np.nan)
    out['sulfur'] = env.get('sulfur', np.nan)
    out['phosphorus'] = env.get('phosphorus', np.nan)
    out['silicon'] = env.get('silicon', np.nan)
    out['iron'] = env.get('iron', np.nan)
    out['magnesium'] = env.get('magnesium', np.nan)
    out['sodium'] = env.get('sodium', np.nan)
    out['potassium'] = env.get('potassium', np.nan)
    out['chlorine'] = env.get('chlorine', np.nan)
    out['methane'] = env.get('methane', np.nan)
    out['ammonia'] = env.get('ammonia', np.nan)
    out['co2'] = env.get('co2', np.nan)

    out['is_extremophile'] = int(env.get('is_extremophile', 0))
    rows.append(out)

df=pd.DataFrame(rows)

# Preserve only rows with sufficient real environmental evidence
required=['temperature_c','ph','salinity_psu','pressure_bar']
df['env_observed_count']=df[ENV_FEATURES].notna().sum(axis=1)
df = df[df['env_observed_count'] >= 4].copy()

# keep robust medians for missing chemistry values (impute with training medians later)
if CFG['max_train_samples']:
    df=df.sample(CFG['max_train_samples'], random_state=SEED).reset_index(drop=True)

print('Rows after environment evidence filter:', len(df))
print(df[['accession','organism','lat','lon','env_observed_count']].head())

## 6) Train/validation split

In [ ]:
if len(df) < 20:
    raise ValueError('Too few rows with observed environmental metadata. Configure BacDive/GOLD connectors and rerun.')

train_df, val_df = train_test_split(
    df,
    test_size=0.15,
    random_state=SEED,
    stratify=df['is_extremophile'] if df['is_extremophile'].nunique()>1 else None
)

# Fit imputation stats on train only
impute_medians = train_df[ENV_FEATURES].median(numeric_only=True)
train_df[ENV_FEATURES] = train_df[ENV_FEATURES].fillna(impute_medians)
val_df[ENV_FEATURES] = val_df[ENV_FEATURES].fillna(impute_medians)

print('train:',train_df.shape,'val:',val_df.shape)
print('train extremophile ratio:',train_df['is_extremophile'].mean())
print('val extremophile ratio:',val_df['is_extremophile'].mean())

## 7) Model + training

In [ ]:
AA_VOCAB=list('ACDEFGHIKLMNPQRSTVWY')+['<pad>','<bos>','<eos>']
a2i={a:i for i,a in enumerate(AA_VOCAB)}; i2a={i:a for a,i in a2i.items()}
PAD,BOS,EOS=a2i['<pad>'],a2i['<bos>'],a2i['<eos>']
class CondSeqDataset(Dataset):
    def __init__(self,frame): self.df=frame.reset_index(drop=True)
    def __len__(self): return len(self.df)
    def __getitem__(self,idx):
        r=self.df.iloc[idx]; toks=[BOS]+[a2i.get(c,PAD) for c in r['sequence'] if c in a2i]+[EOS]
        return {'seq':torch.tensor(toks), 'env':torch.tensor(r[ENV_FEATURES].values.astype(np.float32)), 'is_ext':torch.tensor(r['is_extremophile']).long()}
def collate(batch):
    maxlen=max(len(b['seq']) for b in batch); seqs=torch.full((len(batch),maxlen),PAD,dtype=torch.long)
    for i,b in enumerate(batch): seqs[i,:len(b['seq'])]=b['seq']
    return seqs, torch.stack([b['env'] for b in batch]), torch.stack([b['is_ext'] for b in batch])
class CondDecoder(nn.Module):
    def __init__(self,env_dim,d_model=768,nhead=12,nlayers=6,vocab_size=len(AA_VOCAB)):
        super().__init__(); self.env_proj=nn.Sequential(nn.Linear(env_dim,d_model),nn.GELU(),nn.Linear(d_model,d_model)); self.tok_emb=nn.Embedding(vocab_size,d_model); self.pos_emb=nn.Embedding(2048,d_model)
        layer=nn.TransformerEncoderLayer(d_model=d_model,nhead=nhead,batch_first=True); self.decoder=nn.TransformerEncoder(layer,num_layers=nlayers); self.head=nn.Linear(d_model,vocab_size)
    def forward(self,x,env):
        B,T=x.shape; pos=torch.arange(T,device=x.device).unsqueeze(0).repeat(B,1); h=self.tok_emb(x)+self.pos_emb(pos)+self.env_proj(env).unsqueeze(1)
        mask=torch.triu(torch.ones(T,T,device=x.device),diagonal=1).bool(); return self.head(self.decoder(h,mask=mask))
train_dl=DataLoader(CondSeqDataset(train_df),batch_size=CFG['batch_size'],shuffle=True,num_workers=CFG['num_workers'],collate_fn=collate,pin_memory=True)
val_dl=DataLoader(CondSeqDataset(val_df),batch_size=CFG['batch_size'],shuffle=False,num_workers=CFG['num_workers'],collate_fn=collate,pin_memory=True)
model=CondDecoder(env_dim=len(ENV_FEATURES)).to(DEVICE)
opt=torch.optim.AdamW(model.parameters(),lr=CFG['lr'],weight_decay=CFG['weight_decay'])
criterion=nn.CrossEntropyLoss(ignore_index=PAD)
scaler=torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())
def run_epoch(loader,train=True):
    model.train(train); total=0; n=0
    for step,(seqs,env,is_ext) in enumerate(tqdm(loader)):
        seqs,env=seqs.to(DEVICE),env.to(DEVICE); x=seqs[:,:-1]; y=seqs[:,1:]
        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            logits=model(x,env); loss=criterion(logits.reshape(-1,logits.size(-1)),y.reshape(-1))/CFG['grad_accum']
        if train:
            scaler.scale(loss).backward()
            if (step+1)%CFG['grad_accum']==0: scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
        total += loss.item()*CFG['grad_accum']; n+=1
    return total/max(1,n)
best=float('inf')
for ep in range(1,CFG['epochs']+1):
    tr=run_epoch(train_dl,True); va=run_epoch(val_dl,False); print(f'Epoch {ep}: train={tr:.4f} val={va:.4f}')
    ckpt=Path(CFG['checkpoints_dir'])/f'cond_gen_epoch{ep}.pt'
    torch.save({'model':model.state_dict(),'env_features':ENV_FEATURES,'config':CFG,'epoch':ep},ckpt)
    if va<best: best=va; torch.save({'model':model.state_dict(),'env_features':ENV_FEATURES,'config':CFG,'epoch':ep},Path(CFG['checkpoints_dir'])/'best_cond_gen.pt')

## 8) Generation, validity, extremophile holdout

In [ ]:
VALID_AA=set('ACDEFGHIKLMNPQRSTVWY')
def sample_sequence(env_vector,min_len=100,max_len=256,temperature=0.9,top_p=0.95):
    model.eval(); env=torch.tensor(env_vector,dtype=torch.float32,device=DEVICE).unsqueeze(0); seq=[BOS]
    for _ in range(max_len):
        x=torch.tensor(seq,device=DEVICE).unsqueeze(0)
        with torch.no_grad():
            probs=torch.softmax(model(x,env)[:,-1,:]/temperature,dim=-1)
            sp,si=torch.sort(probs,descending=True); cdf=torch.cumsum(sp,dim=-1); keep=cdf<=top_p; keep[...,0]=True
            fp=torch.zeros_like(probs); fp.scatter_(1,si,sp*keep); fp=fp/fp.sum(dim=-1,keepdim=True); nxt=torch.multinomial(fp,1).item()
        seq.append(nxt)
        if nxt==EOS and len(seq)-2>=min_len: break
    return ''.join(i2a[i] for i in seq[1:] if i2a[i] not in ['<eos>','<pad>','<bos>'])
def seq_validity(s): return int((len(s)>=CFG['min_len']) and (len(s)<=CFG['max_len']) and all(c in VALID_AA for c in s))
example_env=train_df[ENV_FEATURES].iloc[0].values.astype(np.float32)
gen=sample_sequence(example_env,min_len=CFG['min_len'],max_len=CFG['max_len'])
print('Generated length:',len(gen),'valid:',seq_validity(gen))
holdout_ext=val_df[val_df['is_extremophile']==1].copy()
if len(holdout_ext)==0: print('No extremophile holdout examples found; replace with real extremophile dataset.')
else:
    gens=[sample_sequence(r[ENV_FEATURES].values.astype(np.float32),min_len=CFG['min_len'],max_len=CFG['max_len']) for _,r in tqdm(holdout_ext.iterrows(), total=len(holdout_ext))]
    print('Extremophile holdout validity rate:', float(np.mean([seq_validity(s) for s in gens])))

## 9) ESMFold inference hook

In [ ]:
# import esm
# model_fold = esm.pretrained.esmfold_v1().eval().to(DEVICE)
# seq_for_fold = gen[:min(len(gen), 512)]
# with torch.no_grad():
#     pdb_str = model_fold.infer_pdb(seq_for_fold)
# out_pdb = Path(CFG['processed_dir']) / 'generated_fold.pdb'
# with open(out_pdb, 'w') as f: f.write(pdb_str)
# print('Saved:', out_pdb)

## 10) Export artifacts for external UI app

In [ ]:
artifact={'checkpoint_path':str(Path(CFG['checkpoints_dir'])/'best_cond_gen.pt'),'env_features':ENV_FEATURES,'aa_vocab':AA_VOCAB,'min_len_default':CFG['min_len'],'max_len_default':CFG['max_len']}
with open(Path(CFG['checkpoints_dir'])/'artifact_manifest.json','w') as f: json.dump(artifact,f,indent=2)
print(json.dumps(artifact, indent=2))